# Phase 1: Robustness to Class Imbalance (RQ4)

**Objective**: Determine if White-Box and Black-Box representations degrade when trained on highly skewed underlying label distributions. We enforce a strict control baseline to guarantee metric differences are driven exclusively by distribution skew, not sample volume.


# Colab & Environment Setup

In [1]:
!git clone https://github.com/GeorgeVasile04/llm_lie_detection_black_vs_white_box.git

Cloning into 'llm_lie_detection_black_vs_white_box'...
remote: Enumerating objects: 820, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 820 (delta 57), reused 69 (delta 35), pack-reused 724 (from 1)
Receiving objects: 100% (820/820), 54.01 MiB | 18.17 MiB/s, done.
Resolving deltas: 100% (347/347), done.
Updating files: 100% (393/393), done.


In [2]:
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing dependencies...")
    get_ipython().system('pip install -q transformers accelerate datasets scikit-learn pandas')

    from google.colab import drive
    drive.mount('/content/drive')

    import sys, os

    # Check possible paths for the root repo dir
    possible_paths = [
        '/content/drive/MyDrive/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Class_Balance_Impact_BB_WB',
        '/content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Class_Balance_Impact_BB_WB',
        os.path.join(os.getcwd(), 'Black_vs_White_Lie_Detection', 'Class_Balance_Impact_BB_WB'),
        os.getcwd()
    ]

    base_proj_path = None
    for path in possible_paths:
        if os.path.exists(path) and os.path.exists(os.path.join(path, 'load_data_imbalance.py')):
            base_proj_path = path
            break

    if base_proj_path:
        os.chdir(base_proj_path)
        print(f"✅ Changed working directory to: {base_proj_path}")
    else:
        print("⚠️ Could not find project directory automatically. Falling back to simple repo clone approach.")
        # If drive sync fails or the folder isn't found, auto-clone the repo as a fallback!
        get_ipython().system('git clone https://github.com/GeorgeVasile04/llm_lie_detection_black_vs_white_box.git /content/repo')
        base_proj_path = '/content/repo/Black_vs_White_Lie_Detection/Class_Balance_Impact_BB_WB'
        os.chdir(base_proj_path)
        print(f"✅ Downloaded fresh copy from GitHub and moved to: {base_proj_path}")

    # Add Aligned_Comparison_BB_WB to sys path so we can import unmodified core utils
    aligned_path = os.path.abspath(os.path.join(base_proj_path, '..', 'Aligned_Comparison_BB_WB'))
    sys.path.insert(0, aligned_path)
    sys.path.insert(0, base_proj_path)
    # Also add the very root repo to path to ensure repeng loads correctly
    repo_root = os.path.abspath(os.path.join(base_proj_path, '..', '..'))
    sys.path.insert(0, repo_root)

    print(f"✅ Executing in: {os.getcwd()}")

    from huggingface_hub import login
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: login(token=hf_token)

except ImportError:
    IN_COLAB = False
    import sys, os
    # Local fallback
    current_dir = os.getcwd()
    aligned_path = os.path.abspath(os.path.join(current_dir, '..', 'Aligned_Comparison_BB_WB'))
    sys.path.insert(0, aligned_path)
    sys.path.insert(0, current_dir)

Running in Google Colab. Installing dependencies...
Mounted at /content/drive
✅ Changed working directory to: /content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Class_Balance_Impact_BB_WB
✅ Executing in: /content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Class_Balance_Impact_BB_WB


# Imports & Configuration

In [3]:
import torch
import numpy as np
import pandas as pd
import time
from tqdm.auto import tqdm

# Import unmodified feature extractors from Aligned Comparison
from wb_activations import load_model, get_activations_for_dataset
from bb_logprobs import compute_bb_features_for_dataset

# Import customized Imbalance Evaluation Code
from load_data_imbalance import load_dataset_imbalance
from wb_probes_imbalance import train_wb_probes_imbalance
from bb_classifier_imbalance import train_bb_classifier_imbalance
from evaluation_utils_imbalance import compare_results_imbalance

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cuda


# 1. Create Datasets (Scenario A vs Scenario B)

In [ ]:
# We are currently targeting datasets that natively have skewed multiple-choice representations
DATASETS_CONFIG = [
    "commonsense_qa",# 1 True vs 4 False per prompt
    "arc_challenge", # 1 True vs 3 False per prompt
    "open_book_qa",  # 1 True vs 3 False per prompt
    "race",          # 1 True vs 3 False per prompt
    "arc_easy",      # 1 True vs 3 False per prompt
    "ag_news",       # 1 True vs 3 False per prompt
    "dbpedia_14"     # 1 True vs 13 False per prompt
]

dataset_dict = {}

N_SAMPLES = 1000

print("Curating highly controlled class distributions by question ID...\n")
for ds_name in DATASETS_CONFIG:
    print(f"--- Processing {ds_name} ---")

    # Mathematical alignment: dbpedia has 14 options per question. 1000 is not divisible by 14,
    # which would cause the final question to be cut off mid-way. 71 questions * 14 options = 994.
    current_n_samples = 994 if ds_name == "dbpedia_14" else N_SAMPLES

    # Load A and B normally so they both meet their sample volume targets
    df_train_A = load_dataset_imbalance(ds_name, split='train', n_samples=current_n_samples, scenario="A")
    df_train_B = load_dataset_imbalance(ds_name, split='train', n_samples=current_n_samples, scenario="B")

    # Validation: Strictly controlled Balanced (Scenario A logic) to act as the perfect ground-truth exam
    df_val = load_dataset_imbalance(ds_name, split='validation', n_samples=min(current_n_samples, 1000), scenario="A")

    if len(df_train_A) > 0 and len(df_train_B) > 0:

        # Create a perfectly deduplicated union metric to prevent redundant inferences!
        # This keeps total unique row evaluations to the absolute minimum.
        df_train_union = pd.concat([df_train_A, df_train_B]).drop_duplicates(subset=['id', 'text']).reset_index(drop=True)

        dataset_dict[ds_name] = {
            "train_A": df_train_A,
            "train_B": df_train_B,
            "train_union": df_train_union,
            "validation": df_val
        }
        print(f"✅ {ds_name} -> Setup A: {len(df_train_A)} | Setup B: {len(df_train_B)} | UNION (Unique): {len(df_train_union)} | Val: {len(df_val)}\n")
    else:
        print(f"⚠️ {ds_name} failed to load properly. Skipping.")

# Make DATASETS list available for the remaining cells
DATASETS = list(dataset_dict.keys())

Curating highly controlled class distributions by question ID...

--- Processing commonsense_qa ---
Loading commonsense_qa (split=train) for Scenario A, Target N=1000...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dbpedia_14' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dbpedia_14' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


   -> Scenario A: Generated 994 samples (grouped logically).
Loading dbpedia_14 (split=train) for Scenario B, Target N=994...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dbpedia_14' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dbpedia_14' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


   -> Scenario B: Generated 994 samples (grouped logically).
Loading dbpedia_14 (split=validation) for Scenario A, Target N=994...
   -> Scenario A: Generated 994 samples (grouped logically).
✅ dbpedia_14 -> Setup A: 994 | Setup B: 994 | UNION (Unique): 994 | Val: 994



# 2. Load the model

In [ ]:
print("Loading Model...")
model, tokenizer = load_model(MODEL_NAME, device=DEVICE)

Loading Model...
Loading model: meta-llama/Llama-2-7b-chat-hf...


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

# 3. Extract activations vectors for white box

In [ ]:
import gc

WB_BATCH_SIZE = 64 # White Box needs all hidden states, limiting memory

wb_features = {}

for ds_name in tqdm(DATASETS, desc="Dataset WB Extraction"):
    # Dynamically lower batch size heavily for datasets with long text paragraphs!
    if ds_name in ["race"]:
        current_batch_size = 28  # RACE has massive reading comprehension paragraphs
    else:
        current_batch_size = WB_BATCH_SIZE

    df_union = dataset_dict[ds_name]["train_union"]
    df_A = dataset_dict[ds_name]["train_A"]
    df_B = dataset_dict[ds_name]["train_B"]

    # 1. Extract White Box Activations ON THE UNION ONLY
    # This prevents any redundant calculation of overlap!
    act_train_union = get_activations_for_dataset(df_union, model, tokenizer, device=DEVICE, batch_size=current_batch_size, show_progress=True)
    act_val = get_activations_for_dataset(dataset_dict[ds_name]["validation"], model, tokenizer, device=DEVICE, batch_size=current_batch_size, show_progress=True)

    # 2. Map perfectly back to Setup A and Setup B
    # Since each 'text' row in df_union is unique, we pair it using a dictionary mapping
    act_map = {row['text']: act for (_, row), act in zip(df_union.iterrows(), act_train_union)}

    # Instantiate the lists purely by looking up the results matching the DataFrames logic
    act_train_A = [act_map[row['text']] for _, row in df_A.iterrows()]
    act_train_B = [act_map[row['text']] for _, row in df_B.iterrows()]

    wb_features[ds_name] = {"train_A": act_train_A, "train_B": act_train_B, "validation": act_val}
    print(f"✅ {ds_name} WB -> Union inferences: {len(act_train_union)} physically computed. Mapped to A({len(act_train_A)}) & B({len(act_train_B)})")

    # 🧹 Thoroughly clean GPU and RAM memory after each dataset
    torch.cuda.empty_cache()
    gc.collect()

Dataset WB Extraction:   0%|          | 0/7 [00:00<?, ?it/s]

Extracting activations (batch_size=64):   0%|          | 0/25 [00:00<?, ?it/s]

Extracting activations (batch_size=64):   0%|          | 0/16 [00:00<?, ?it/s]

✅ commonsense_qa WB -> Union inferences: 1595 physically computed. Mapped to A(1000) & B(1000)


Extracting activations (batch_size=64):   0%|          | 0/24 [00:00<?, ?it/s]

...

Extracting activations (batch_size=64):   0%|          | 0/16 [00:00<?, ?it/s]

✅ ag_news WB -> Union inferences: 1000 physically computed. Mapped to A(1000) & B(1000)


Extracting activations (batch_size=64):   0%|          | 0/16 [00:00<?, ?it/s]

Extracting activations (batch_size=64):   0%|          | 0/16 [00:00<?, ?it/s]

✅ dbpedia_14 WB -> Union inferences: 994 physically computed. Mapped to A(994) & B(994)


# 4. Compute logprobs for black box

In [ ]:
import os
import gc

# Load the exact elicitation questions used in the original Black Box paper
try:
    # We navigate from Class_Balance_Impact_BB_WB to the exact folder
    probe_base = "../../Black_Box_Lie_Detection/results/probes_groups"
    no_lie_indices = np.load(os.path.join(probe_base, "no_lie_indices.npy"))
    lie_indices = np.load(os.path.join(probe_base, "lie_indices.npy"))
    knowable_indices = np.load(os.path.join(probe_base, "knowable_indices.npy"))
    PAPER_PROBE_INDICES = np.concatenate([no_lie_indices, lie_indices, knowable_indices])
    print(f"✅ Loaded {len(PAPER_PROBE_INDICES)} elicitation questions (exact same as original paper)")
except FileNotFoundError as e:
    print(f"⚠️ Warning: Could not load original paper's probe indices: {e}")
    print("Will use all available probes instead.")
    PAPER_PROBE_INDICES = None

BB_BATCH_SIZE = 480 # Black Box only needs logits, so we can stack much larger batches

bb_features = {}
for ds_name in tqdm(DATASETS, desc="Dataset BB Extraction"):
    df_union = dataset_dict[ds_name]["train_union"]
    df_A = dataset_dict[ds_name]["train_A"]
    df_B = dataset_dict[ds_name]["train_B"]

    # 1. Compute Black Box Logprobs ON THE UNION ONLY
    bb_train_union = compute_bb_features_for_dataset(
        df_union, model, tokenizer, device=DEVICE, batch_size=BB_BATCH_SIZE, probe_indices=PAPER_PROBE_INDICES, show_progress=True
    )
    bb_val = compute_bb_features_for_dataset(
        dataset_dict[ds_name]["validation"], model, tokenizer, device=DEVICE, batch_size=BB_BATCH_SIZE, probe_indices=PAPER_PROBE_INDICES, show_progress=True
    )

    # 2. Map perfectly back to A and B
    bb_map = {row['text']: bb for (_, row), bb in zip(df_union.iterrows(), bb_train_union)}

    bb_train_A = [bb_map[row['text']] for _, row in df_A.iterrows()]
    bb_train_B = [bb_map[row['text']] for _, row in df_B.iterrows()]

    # Unpack BB to numpy matrices aligned precisely
    def unpack_bb(bb_dict):
        X, Y = [], []
        for r in bb_dict:
            if "bb_features" in r and r["bb_features"]:
                X.append([p["score"] for p in r["bb_features"]])
                Y.append(r["label"])
        return np.array(X), np.array(Y)

    bb_features[ds_name] = {
        "train_A": unpack_bb(bb_train_A),
        "train_B": unpack_bb(bb_train_B),
        "validation": unpack_bb(bb_val)
    }

    # 🧹 Thoroughly clean GPU and RAM memory after each dataset
    torch.cuda.empty_cache()
    gc.collect()

✅ Loaded 48 elicitation questions (exact same as original paper)


Dataset BB Extraction:   0%|          | 0/7 [00:00<?, ?it/s]

Computing BB features (batch_size=480):   0%|          | 0/4 [00:00<?, ?it/s]

Computing BB features (batch_size=480):   0%|          | 0/3 [00:00<?, ?it/s]

Computing BB features (batch_size=480):   0%|          | 0/4 [00:00<?, ?it/s]

...

Computing BB features (batch_size=480):   0%|          | 0/3 [00:00<?, ?it/s]

Computing BB features (batch_size=480):   0%|          | 0/3 [00:00<?, ?it/s]

Computing BB features (batch_size=480):   0%|          | 0/3 [00:00<?, ?it/s]

Computing BB features (batch_size=480):   0%|          | 0/3 [00:00<?, ?it/s]

Computing BB features (batch_size=480):   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
import pickle
import os

checkpoint_dir = '/content/drive/MyDrive/llm_lie_detection_black_vs_white_box/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# ==========================================
# 💾 TO SAVE BEFORE SHUTTING DOWN (COMMENTED OUT TO PREVENT ACCIDENTAL OVERWRITES)
# ==========================================
# print("Saving White-Box features to Google Drive...")
# with open(os.path.join(checkpoint_dir, 'wb_features.pkl'), 'wb') as f:
#     pickle.dump(wb_features, f)
#
# print("Saving Black-Box features to Google Drive...")
# with open(os.path.join(checkpoint_dir, 'bb_features.pkl'), 'wb') as f:
#     pickle.dump(bb_features, f)
#
# print(f"✅ Successfully saved expensive intermediate features to {checkpoint_dir}!")

# ==========================================
# 🔄 WHEN YOU COME BACK: RUN THIS TO SKIP STEPS 1-4
# ==========================================
print("Loading White-Box features from Google Drive...")
with open(os.path.join(checkpoint_dir, 'wb_features.pkl'), 'rb') as f:
    wb_features = pickle.load(f)

print("Loading Black-Box features from Google Drive...")
with open(os.path.join(checkpoint_dir, 'bb_features.pkl'), 'rb') as f:
    bb_features = pickle.load(f)

DATASETS = list(wb_features.keys())
print("✅ Successfully loaded features! You can now proceed directly to Step 5.")

Loading White-Box features from Google Drive...
Loading Black-Box features from Google Drive...
✅ Successfully loaded features! You can now proceed directly to Step 5.


# 5. Train Classifiers and Expose Imbalance Impact

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, precision_recall_curve
from scipy.stats import wilcoxon
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import os
import sys
from contextlib import redirect_stdout, redirect_stderr

def best_recall_at_precision(y_true, y_scores, target_precision=0.90):
    """Calculates best recall maintaining at least 'target_precision'"""
    if len(np.unique(y_true)) < 2:
        return 0.0
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)
    # Find all indices where precision >= target_precision
    valid_idx = np.where(precisions >= target_precision)[0]
    if len(valid_idx) == 0:
        return 0.0  # No threshold gives required precision
    return np.max(recalls[valid_idx])

from wb_probes_imbalance import select_best_layer_cv

# Expand Layer List to include ALL 32 layers for the Cross-Validation step
LAYER_LIST_CV = list(range(1, 33))

# Include all 8 White-Box probe variations available in wb_probes.py
WB_METHODS = ["lr", "dim", "lat", "ccs", "lda", "pca", "pca-g", "lr-g"]

all_results = []

for ds_name in DATASETS:
    print(f"\n--- Evaluating {ds_name} on FULL 1K Test Set ---")

    # Data Pointers
    w_tr_A, w_tr_B, w_v = wb_features[ds_name]["train_A"], wb_features[ds_name]["train_B"], wb_features[ds_name]["validation"]
    b_tr_A, b_tr_B, b_v = bb_features[ds_name]["train_A"], bb_features[ds_name]["train_B"], bb_features[ds_name]["validation"]
    X_val_bb, y_val_bb = b_v

    def calculate_full_metrics(probs_or_logits):
        auc = roc_auc_score(y_val_bb, probs_or_logits) if len(np.unique(y_val_bb)) > 1 else 0.5
        map_score = average_precision_score(y_val_bb, probs_or_logits) if len(np.unique(y_val_bb)) > 1 else 0.5
        brp_90 = best_recall_at_precision(y_val_bb, probs_or_logits, target_precision=0.90)
        return {'AUC': auc, 'MAP': map_score, 'BRP_90': brp_90}

    # Progress bar tracks BB and WB training for both A and B (2 arrays * (1 BB + 8 WB) = 18 total steps)
    pbar = tqdm(total=2 * (1 + len(WB_METHODS)), desc=f"Training {ds_name}")

    with open(os.devnull, 'w') as f, redirect_stdout(f), redirect_stderr(f):
        bb_model_A, bb_met_A_full = train_bb_classifier_imbalance(b_tr_A[0], X_val_bb, b_tr_A[1], y_val_bb, verbose=False)
    pbar.update(1)

    wb_met_A_full = {}
    for method in WB_METHODS:
        # Cross-validation layer selection over ALL 32 layers
        best_layer_A = select_best_layer_cv(w_tr_A, layer_list=LAYER_LIST_CV, method=method, verbose=False)
        with open(os.devnull, 'w') as f, redirect_stdout(f), redirect_stderr(f):
            _, wb_met = train_wb_probes_imbalance(w_tr_A, eval_data=w_v, layer_list=[best_layer_A], method=method, verbose=False)
        wb_met_A_full[method] = wb_met
        wb_met_A_full[method]['best_layer'] = best_layer_A
        pbar.update(1)

    with open(os.devnull, 'w') as f, redirect_stdout(f), redirect_stderr(f):
        bb_model_B, bb_met_B_full = train_bb_classifier_imbalance(b_tr_B[0], X_val_bb, b_tr_B[1], y_val_bb, verbose=False)
    pbar.update(1)

    wb_met_B_full = {}
    for method in WB_METHODS:
        # Cross-validation layer selection over ALL 32 layers
        best_layer_B = select_best_layer_cv(w_tr_B, layer_list=LAYER_LIST_CV, method=method, verbose=False)
        with open(os.devnull, 'w') as f, redirect_stdout(f), redirect_stderr(f):
            _, wb_met = train_wb_probes_imbalance(w_tr_B, eval_data=w_v, layer_list=[best_layer_B], method=method, verbose=False)
        wb_met_B_full[method] = wb_met
        wb_met_B_full[method]['best_layer'] = best_layer_B
        pbar.update(1)

    pbar.close()

    # Evaluate BB
    bb_met_A = calculate_full_metrics(bb_met_A_full["probs"])
    bb_met_B = calculate_full_metrics(bb_met_B_full["probs"])

    all_results.append({"Dataset": ds_name, "Condition": "Setup A (Balanced)", "Method": "Black Box", "Algorithm": "BB", "Layer": "N/A", "AUC": bb_met_A["AUC"], "MAP": bb_met_A["MAP"], "BRP_90": bb_met_A["BRP_90"]})
    all_results.append({"Dataset": ds_name, "Condition": "Setup B (Imbalanced)", "Method": "Black Box", "Algorithm": "BB", "Layer": "N/A", "AUC": bb_met_B["AUC"], "MAP": bb_met_B["MAP"], "BRP_90": bb_met_B["BRP_90"]})

    # Evaluate WB
    for method in WB_METHODS:
        best_l_a = wb_met_A_full[method]['best_layer']
        if best_l_a in wb_met_A_full[method]:
            mA = calculate_full_metrics(wb_met_A_full[method][best_l_a]["logits"])
            all_results.append({"Dataset": ds_name, "Condition": "Setup A (Balanced)", "Method": "White Box", "Algorithm": method.upper(), "Layer": best_l_a, "AUC": mA["AUC"], "MAP": mA["MAP"], "BRP_90": mA["BRP_90"]})

        best_l_b = wb_met_B_full[method]['best_layer']
        if best_l_b in wb_met_B_full[method]:
            mB = calculate_full_metrics(wb_met_B_full[method][best_l_b]["logits"])
            all_results.append({"Dataset": ds_name, "Condition": "Setup B (Imbalanced)", "Method": "White Box", "Algorithm": method.upper(), "Layer": best_l_b, "AUC": mB["AUC"], "MAP": mB["MAP"], "BRP_90": mB["BRP_90"]})

final_df = pd.DataFrame(all_results)
final_df.to_csv("class_imbalance_metrics_detailed.csv", index=False)
print("\n✅ Saved detailed metric evaluations per layer to class_imbalance_metrics_detailed.csv")


--- Evaluating commonsense_qa on FULL 1K Test Set ---


Training commonsense_qa:   0%|          | 0/18 [00:00<?, ?it/s]


--- Evaluating arc_challenge on FULL 1K Test Set ---


Training arc_challenge:   0%|          | 0/18 [00:00<?, ?it/s]


--- Evaluating open_book_qa on FULL 1K Test Set ---


...


--- Evaluating ag_news on FULL 1K Test Set ---


Training ag_news:   0%|          | 0/18 [00:00<?, ?it/s]


--- Evaluating dbpedia_14 on FULL 1K Test Set ---


Training dbpedia_14:   0%|          | 0/18 [00:00<?, ?it/s]


✅ Saved detailed metric evaluations per layer to class_imbalance_metrics_detailed.csv


# 6. View Imbalance Degradation Metrics


In [6]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from IPython.display import display

# Ensure dataframe is loaded (if running this cell independently)
if 'final_df' not in locals():
    final_df = pd.read_csv("class_imbalance_metrics_detailed.csv")

metrics_to_test = ["AUC", "MAP", "BRP_90"]
wilcoxon_results = []

# For each metric, we pick the best layer indepedently for A and for B
for metric in metrics_to_test:
    best_A = []
    best_B = []
    pairs_data = [] # Store detailed pairs for the current metric

    for ds_name in final_df["Dataset"].unique():
        for algo in final_df["Algorithm"].unique():
            sub_df = final_df[(final_df["Dataset"] == ds_name) & (final_df["Algorithm"] == algo)]

            sub_A = sub_df[sub_df["Condition"] == "Setup A (Balanced)"]
            sub_B = sub_df[sub_df["Condition"] == "Setup B (Imbalanced)"]

            if not sub_A.empty and not sub_B.empty:
                # Select the max value for that specific metric
                val_A = sub_A[metric].max()
                val_B = sub_B[metric].max()

                best_A.append(val_A)
                best_B.append(val_B)

                # Record the pair comparison
                pairs_data.append({
                    "Dataset": ds_name,
                    "Algorithm": algo,
                    f"Setup A (Balanced) {metric}": val_A,
                    f"Setup B (Imbalanced) {metric}": val_B,
                    "Difference (A - B)": val_A - val_B
                })

    # Perform Wilcoxon Signed-Rank Test
    stat, p_val = wilcoxon(best_A, best_B)

    # Positive mean_diff means A > B (Balanced > Imbalanced)
    mean_diff = np.mean(np.array(best_A) - np.array(best_B))

    # Determine general conclusion
    if p_val < 0.05:
        if mean_diff > 0:
            conclusion = "A > B (Imbalance degrades performace)"
        else:
            conclusion = "B > A (Imbalance improves performance)"
    else:
        conclusion = "No significant difference"

    wilcoxon_results.append({
        "Metric": metric,
        "Pairs Count": len(best_A),
        "Mean Diff (A - B)": round(mean_diff, 4),
        "W-Statistic": stat,
        "P-Value": p_val,
        "Significant": p_val < 0.05,
        "Conclusion": conclusion
    })

    # Save the 63 aligned pairs specifically for this metric
    pairs_df = pd.DataFrame(pairs_data)
    file_name = f"wilcoxon_pairs_{metric}.csv"
    pairs_df.to_csv(file_name, index=False)
    print(f"✅ Saved {len(best_A)} paired {metric} comparisons to {file_name}")

wilcoxon_df = pd.DataFrame(wilcoxon_results)

print("\n🌟 PHASE 1 EXPERIMENT COMPLETE: CLASS IMBALANCE WILCOXON TESTS 🌟")
print("These tests optimize the best layer per algorithm for each metric explicitly.")
print("Comparing 63 dimensions (7 datasets x 9 classifiers).\n")

display(wilcoxon_df)

wilcoxon_df.to_csv("wilcoxon_global_summary.csv", index=False)
print("\n✅ Saved global Wilcoxon Signed-Rank summary testing to wilcoxon_global_summary.csv")

✅ Saved 63 paired AUC comparisons to wilcoxon_pairs_AUC.csv
✅ Saved 63 paired MAP comparisons to wilcoxon_pairs_MAP.csv
✅ Saved 63 paired BRP_90 comparisons to wilcoxon_pairs_BRP_90.csv

🌟 PHASE 1 EXPERIMENT COMPLETE: CLASS IMBALANCE WILCOXON TESTS 🌟
These tests optimize the best layer per algorithm for each metric explicitly.
Comparing 63 dimensions (7 datasets x 9 classifiers).



,Metric,Pairs Count,Mean Diff (A - B),W-Statistic,P-Value,Significant,Conclusion
0,AUC,63,0.0044,224.0,0.000111,True,A > B (Imbalance degrades performace)
1,MAP,63,0.0042,253.0,0.000349,True,A > B (Imbalance degrades performace)
2,BRP_90,63,0.0012,319.5,0.098831,False,No significant difference



✅ Saved global Wilcoxon Signed-Rank summary testing to wilcoxon_global_summary.csv
